In [0]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog
from pyspark.sql import functions as F
 
BATCH_SIZE = 500
BUDGET_HEADROOM_PCT = 0.30
MAX_BUFFER_MULTIPLIER = 0.5
 
print("--- Starting End-to-End Supply Chain Optimization ---")
 

--- Starting End-to-End Supply Chain Optimization ---


In [0]:
# 1. Pull the latest Gold table, select this cycle's candidates deterministically
inventory_gold = spark.table("supply_chain_opt.gold_inventory_master")
 
optimization_candidates = inventory_gold \
    .filter("abc_category = 'A' AND priority_level = 'CRITICAL'") \
    .orderBy(F.desc("stock_out_risk_score")) \
    .limit(BATCH_SIZE) \
    .toPandas()

In [0]:
if not optimization_candidates.empty:
    print(f"Optimizing {len(optimization_candidates)} of the highest-risk Category-A CRITICAL items...")
 
    data = optimization_candidates
    data['shortfall'] = (data['reorder_point_adj'] - data['current_stock']).clip(lower=0)
    data['upper_limit'] = data['shortfall'] + MAX_BUFFER_MULTIPLIER * data['reorder_point_adj']
    bounds = list(zip(data['shortfall'], data['upper_limit']))
 
    min_cost = float((data['shortfall'] * data['unit_cost']).sum())
    max_cost = float((data['upper_limit'] * data['unit_cost']).sum())
    budget_limit = min_cost + BUDGET_HEADROOM_PCT * (max_cost - min_cost)
 
    # Objective: maximize risk-weighted spend (put the marginal dollar where risk drops the most)
    c = data['stock_out_risk_score'].values * -1
    A_ub = [data['unit_cost'].values]
    b_ub = [budget_limit]
 
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
 
    if res.success:
        data['optimized_order_qty'] = np.round(res.x)
        data['total_investment'] = data['optimized_order_qty'] * data['unit_cost']
 
        # Sanity check: guard against the old degenerate "everything into one SKU" failure mode.
        # This is exactly the bug the previous version had (~63% of budget into a single item) -
        # if this ever trips again, the bounds/budget logic above needs a second look before trusting
        # the output.
        max_share = data['total_investment'].max() / data['total_investment'].sum()
        assert max_share < 0.10, f"Solver concentrated {max_share:.1%} of spend in a single SKU - check bounds."
 
        # gold_inventory_master carries BOTH the original `reorder_point` and the vendor-adjusted
        # `reorder_point_adj` (kept for auditability in Phase 2). Drop the original before renaming
        # the adjusted one, or pandas silently produces two columns named `reorder_point` and
        # Spark's createDataFrame correctly rejects it with COLUMN_ALREADY_EXISTS.
        final_df = data.drop(columns=["reorder_point"]).rename(columns={"reorder_point_adj": "reorder_point"})
        final_spark_df = spark.createDataFrame(final_df)
        final_spark_df.write.mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable("supply_chain_opt.final_procurement_plan")
 
        print(f"Success! Budget used: ${data['total_investment'].sum():,.2f} of ${budget_limit:,.2f}")
        print(f"Largest single-item share of spend: {max_share:.1%} (sanity check passed)")
    else:
        print(f"Solver Error: {res.message}. Try widening the budget headroom or the batch size.")
else:
    print("No critical Category-A items found this cycle. Inventory is within stable limits.")
 
print("--- Pipeline Execution Complete ---")
 

Optimizing 500 of the highest-risk Category-A CRITICAL items...
Success! Budget used: $129,395,162.56 of $129,396,047.57
Largest single-item share of spend: 0.5% (sanity check passed)
--- Pipeline Execution Complete ---
